# Preprocessing Remove Utilities Demo

Demostración interactiva de las funciones de depuración y eliminación de columnas redundantes o invariables implementadas en `_preprocessing_removes.py` y expuestas por `ds_utils.preprocessing.preprocessing_utilities`.

Funciones cubiertas en este notebook:
- `remove_redundant_sensor_noise_columns`
- `remove_constant_columns`
- `remove_replaceable_columns`

## Imports

In [1]:
# RECARGA AUTOMÁTICA DE MÓDULOS EXTERNOS
%load_ext autoreload
%autoreload 2

import pandas as pd

from ds_utils.preprocessing.preprocessing_utilities import (
    remove_redundant_sensor_noise_columns,
    remove_constant_columns,
    remove_replaceable_columns,
    handle_sensor_noise,  # utilizado para preparar datos de ruido de sensor
)

## remove_redundant_sensor_noise_columns

`remove_redundant_sensor_noise_columns` elimina las columnas derivadas de ruido (`_clean`, `_is_noise`, etc.) para aquellos sensores cuya señal resultó totalmente limpia y no requirió ninguna corrección.

In [2]:
# 1. Generamos dos señales: una limpia y una con ruido
df_sensors_raw = pd.DataFrame({
    "sensor_clean": [10.0, 10.1, 10.2, 10.1, 10.0],
    "sensor_noisy": [20.0, 20.1, 99.9, 20.2, 20.0],
})

tolerances = {"sensor_clean": 2.0, "sensor_noisy": 2.0}
df_with_noise_cols, _ = handle_sensor_noise(df=df_sensors_raw, tolerances=tolerances)

print("Columnas antes de eliminar redundancias:")
print(list(df_with_noise_cols.columns))

# 2. Eliminación de derivadas redundantes
df_noise_pruned, removed_noise_cols = remove_redundant_sensor_noise_columns(
    df=df_with_noise_cols,
    tolerances=tolerances
)

print("Columnas eliminadas por redundancia:", removed_noise_cols)
print("\nDataFrame resultante (se mantienen solo las derivadas del sensor ruidoso):")
display(df_noise_pruned)

Columnas antes de eliminar redundancias:
['sensor_clean', 'sensor_noisy', 'sensor_clean_clean', 'sensor_clean_is_noise', 'sensor_noisy_clean', 'sensor_noisy_is_noise', 'any_sensor_noise']
Columnas eliminadas por redundancia: ['sensor_clean_clean', 'sensor_clean_is_noise']

DataFrame resultante (se mantienen solo las derivadas del sensor ruidoso):


,sensor_clean,sensor_noisy,sensor_noisy_clean,sensor_noisy_is_noise,any_sensor_noise
0,10.0,20.0,20.0,False,False
1,10.1,20.1,20.1,False,False
2,10.2,99.9,20.1,True,True
3,10.1,20.2,20.2,False,False
4,10.0,20.0,20.0,False,False


## remove_constant_columns

`remove_constant_columns` elimina columnas con un único valor invariable (`nunique == 1`), permitiendo excluir columnas protegidas (`excluded_columns`).

In [3]:
# 1. DataFrame con constantes y variables
df_const_sample = pd.DataFrame({
    "id": [1, 2, 3, 4],
    "fixed_status": ["OK", "OK", "OK", "OK"],
    "fixed_value": [0.0, 0.0, 0.0, 0.0],
    "protected_flag": [True, True, True, True],
    "feature": [10.5, 20.3, 15.8, 30.1]
})

# 2. Eliminación protegiendo 'protected_flag'
df_no_const, removed_const_cols = remove_constant_columns(
    df=df_const_sample,
    excluded_columns=["protected_flag"]
)

print("Columnas eliminadas:", removed_const_cols)
print("\nDataFrame sin constantes:")
display(df_no_const)

Columnas eliminadas: ['fixed_status', 'fixed_value']

DataFrame sin constantes:


,id,protected_flag,feature
0,1,True,10.5
1,2,True,20.3
2,3,True,15.8
3,4,True,30.1


## remove_replaceable_columns

`remove_replaceable_columns` elimina las columnas base u originales cuando ya existe una versión de reemplazo identificada por un prefijo o sufijo (por ejemplo eliminando `temp` al existir `temp_clean`).

In [4]:
# 1. DataFrame con señales originales y versiones limpias
df_replaceable = pd.DataFrame({
    "temperature": [20.0, 95.0, 20.2],
    "temperature_clean": [20.0, 20.1, 20.2],
    "pressure": [1.0, 1.1, 1.0],
    "pressure_clean": [1.0, 1.0, 1.0],
    "target": [0, 1, 0]
})

# 2. Eliminación de las versiones originales por sufijo '_clean'
df_pruned, removed_originals = remove_replaceable_columns(
    df=df_replaceable,
    replacement_suffix="_clean"
)

print("Columnas originales sustituidas y eliminadas:", removed_originals)
print("\nDataFrame resultante:")
display(df_pruned)

Columnas originales sustituidas y eliminadas: ['pressure', 'temperature']

DataFrame resultante:


,temperature_clean,pressure_clean,target
0,20.0,1.0,0
1,20.1,1.0,1
2,20.2,1.0,0
